# Day 3 — Statistical Outlier Detection

## Objective

Today I will use the feature-engineered dataset from Day 2
and detect statistical outliers using:

- Z-Score
- Interquartile Range (IQR)
- Pandas
- NumPy
- SciPy

The implementation will use vectorized operations instead of
row-wise Python loops.

In [26]:
import pandas as pd
import numpy as np
from scipy import stats

In [27]:
df = pd.read_csv("../Day 2/features_v1.csv")

df.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,...,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess
0,1,0,3,22.0,1.0,0,7.2500,2.0,0,3.6250,...,0,0,0,0,1,0,0,0,0,0
1,2,1,1,38.0,1.0,0,65.6344,2.0,0,32.8172,...,0,0,0,0,0,1,0,0,0,0
2,3,1,3,26.0,0.0,0,7.9250,1.0,1,7.9250,...,0,1,0,0,0,0,0,0,0,0
3,4,1,1,35.0,1.0,0,53.1000,2.0,0,26.5500,...,0,0,0,0,0,1,0,0,0,0
4,5,0,3,35.0,0.0,0,8.0500,1.0,1,8.0500,...,0,0,0,0,1,0,0,0,0,0


In [28]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (891, 29)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Sex_Male', 'Embarked_Q', 'Embarked_S', 'Title_Col', 'Title_Don', 'Title_Dr', 'Title_Jonkheer', 'Title_Lady', 'Title_Major', 'Title_Master', 'Title_Miss', 'Title_Mlle', 'Title_Mme', 'Title_Mr', 'Title_Mrs', 'Title_Ms', 'Title_Rev', 'Title_Sir', 'Title_the Countess']

Data Types:
PassengerId             int64
Survived                int64
Pclass                  int64
Age                   float64
SibSp                 float64
Parch                   int64
Fare                  float64
FamilySize            float64
IsAlone                 int64
FarePerPerson         float64
Sex_Male                int64
Embarked_Q              int64
Embarked_S              int64
Title_Col               int64
Title_Don               int64
Title_Dr                int64
Title_Jonkheer          int64
Title_Lady              int64
Title_Major             int64
Ti

In [29]:
numeric_columns = df.select_dtypes(include=np.number).columns

print("Numerical columns:")
print(numeric_columns.tolist())

Numerical columns:
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Sex_Male', 'Embarked_Q', 'Embarked_S', 'Title_Col', 'Title_Don', 'Title_Dr', 'Title_Jonkheer', 'Title_Lady', 'Title_Major', 'Title_Master', 'Title_Miss', 'Title_Mlle', 'Title_Mme', 'Title_Mr', 'Title_Mrs', 'Title_Ms', 'Title_Rev', 'Title_Sir', 'Title_the Countess']


## 3. Z-Score Outlier Detection

The Z-Score measures how far a value is from the mean in terms of
standard deviations.

A Z-Score greater than 3 or less than -3 is considered a potential
outlier.

The calculation is performed using SciPy and vectorized operations.

In [30]:
z_scores = np.abs(
    stats.zscore(df[numeric_columns], nan_policy="omit")
)

z_scores = pd.DataFrame(
    z_scores,
    columns=numeric_columns,
    index=df.index
)

z_scores.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,...,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess
0,1.730108,0.789272,0.827377,0.583432,0.810220,NaN,0.820552,0.810220,1.465746,0.882136,...,0.216803,0.506655,0.047431,0.03352,0.850532,0.403962,0.03352,0.082339,0.03352,0.03352
1,1.726220,1.266990,1.566107,0.742685,0.810220,NaN,2.031623,0.810220,1.465746,0.933715,...,0.216803,0.506655,0.047431,0.03352,1.175735,2.475480,0.03352,0.082339,0.03352,0.03352
2,1.722332,1.266990,0.827377,0.251903,0.602512,NaN,0.787578,0.602512,0.682247,0.614662,...,0.216803,1.973729,0.047431,0.03352,1.175735,0.403962,0.03352,0.082339,0.03352,0.03352
3,1.718444,1.266990,1.566107,0.494038,0.810220,NaN,1.419297,0.810220,1.465746,0.543875,...,0.216803,0.506655,0.047431,0.03352,1.175735,2.475480,0.03352,0.082339,0.03352,0.03352
4,1.714556,0.789272,0.827377,0.494038,0.602512,NaN,0.781471,0.602512,0.682247,0.606886,...,0.216803,0.506655,0.047431,0.03352,0.850532,0.403962,0.03352,0.082339,0.03352,0.03352


In [31]:
z_score_mask = (z_scores > 3).any(axis=1)

print("Number of Z-Score outlier rows:", z_score_mask.sum())

Number of Z-Score outlier rows: 137


In [32]:
z_score_outliers = df.loc[z_score_mask]

print("Z-Score outlier rows:")
print(z_score_outliers.shape)

display(z_score_outliers.head(10))

Z-Score outlier rows:
(137, 29)


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,...,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess
5,6,0,3,28.0,0.0,0,8.4583,1.0,1,8.458300,...,0,0,0,0,1,0,0,0,0,0
7,8,0,3,2.5,2.5,0,21.0750,3.5,0,6.021429,...,1,0,0,0,0,0,0,0,0,0
16,17,0,3,2.5,2.5,0,29.1250,3.5,0,8.321429,...,1,0,0,0,0,0,0,0,0,0
22,23,1,3,15.0,0.0,0,8.0292,1.0,1,8.029200,...,0,1,0,0,0,0,0,0,0,0
28,29,1,3,28.0,0.0,0,7.8792,1.0,1,7.879200,...,0,1,0,0,0,0,0,0,0,0
30,31,0,1,40.0,0.0,0,27.7208,1.0,1,27.720800,...,0,0,0,0,0,0,0,0,0,0
32,33,1,3,28.0,0.0,0,7.7500,1.0,1,7.750000,...,0,1,0,0,0,0,0,0,0,0
44,45,1,3,19.0,0.0,0,7.8792,1.0,1,7.879200,...,0,1,0,0,0,0,0,0,0,0
46,47,0,3,28.0,1.0,0,15.5000,2.0,0,7.750000,...,0,0,0,0,1,0,0,0,0,0
47,48,1,3,28.0,0.0,0,7.7500,1.0,1,7.750000,...,0,1,0,0,0,0,0,0,0,0


## 4. IQR Outlier Detection

The Interquartile Range (IQR) is calculated using the 25th percentile
(Q1) and 75th percentile (Q3).

IQR = Q3 - Q1

The lower and upper boundaries are:

Lower Bound = Q1 - 1.5 × IQR
Upper Bound = Q3 + 1.5 × IQR

Values outside these boundaries are considered potential outliers.

In [33]:
Q1 = df[numeric_columns].quantile(0.25)
Q3 = df[numeric_columns].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("IQR calculation completed.")

IQR calculation completed.


In [34]:
iqr_outlier_mask = (
    (df[numeric_columns] < lower_bound) |
    (df[numeric_columns] > upper_bound)
).any(axis=1)

print("Number of IQR outlier rows:", iqr_outlier_mask.sum())

Number of IQR outlier rows: 435


## 5. Handling Outliers

The IQR Boolean mask is used to identify rows containing outliers.

Rows identified as IQR outliers are removed using Pandas Boolean
indexing. This approach is vectorized and does not use row-wise
Python for-loops.

In [35]:
df_clean = df.loc[~iqr_outlier_mask].copy()

print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)

Original dataset shape: (891, 29)
Cleaned dataset shape: (456, 29)


In [36]:
print("Missing values after outlier handling:")
print(df_clean.isnull().sum())

print("\nCleaned dataset preview:")
display(df_clean.head())

Missing values after outlier handling:
PassengerId           0
Survived              0
Pclass                0
Age                   0
SibSp                 0
Parch                 0
Fare                  0
FamilySize            0
IsAlone               0
FarePerPerson         0
Sex_Male              0
Embarked_Q            0
Embarked_S            0
Title_Col             0
Title_Don             0
Title_Dr              0
Title_Jonkheer        0
Title_Lady            0
Title_Major           0
Title_Master          0
Title_Miss            0
Title_Mlle            0
Title_Mme             0
Title_Mr              0
Title_Mrs             0
Title_Ms              0
Title_Rev             0
Title_Sir             0
Title_the Countess    0
dtype: int64

Cleaned dataset preview:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,...,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess
0,1,0,3,22.0,1.0,0,7.2500,2.0,0,3.6250,...,0,0,0,0,1,0,0,0,0,0
4,5,0,3,35.0,0.0,0,8.0500,1.0,1,8.0500,...,0,0,0,0,1,0,0,0,0,0
6,7,0,1,54.0,0.0,0,51.8625,1.0,1,51.8625,...,0,0,0,0,1,0,0,0,0,0
12,13,0,3,20.0,0.0,0,8.0500,1.0,1,8.0500,...,0,0,0,0,1,0,0,0,0,0
13,14,0,3,39.0,1.0,0,31.2750,2.0,0,15.6375,...,0,0,0,0,1,0,0,0,0,0


## 6. Exporting the Cleaned Dataset

The final cleaned dataset is exported as `final_clean_v2.csv`
for submission and further use.

In [37]:
output_path = "final_clean_v2.csv"

df_clean.to_csv(output_path, index=False)

print("Final cleaned dataset saved successfully!")
print("File:", output_path)

Final cleaned dataset saved successfully!
File: final_clean_v2.csv


In [38]:
print("Number of Z-Score outlier rows:", z_score_mask.sum())


Number of Z-Score outlier rows: 137


In [39]:
print("Number of IQR outlier rows:", iqr_outlier_mask.sum())

Number of IQR outlier rows: 435


In [40]:
print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)

Original dataset shape: (891, 29)
Cleaned dataset shape: (456, 29)


In [41]:
print("===== DAY 3 FINAL SUMMARY =====")
print("Original dataset:", df.shape)
print("Z-Score outliers:", z_score_mask.sum())
print("IQR outliers:", iqr_outlier_mask.sum())
print("Final cleaned dataset:", df_clean.shape)
print("Output file: final_clean_v2.csv")

===== DAY 3 FINAL SUMMARY =====
Original dataset: (891, 29)
Z-Score outliers: 137
IQR outliers: 435
Final cleaned dataset: (456, 29)
Output file: final_clean_v2.csv
